# FingerHut submission 4 Python + Prophet pipeline

Python translation of Charlie's `01_data_cleaning.qmd`, `02_sampling_flattening.qmd`, and `03_modelling.qmd` workflow, with Prophet order-volume forecast components added as model features. It uses Polars for the data.table-style large event processing, Prophet for daily shipped-order seasonality, and scikit-learn/XGBoost for the final classifier.

Outputs are written next to this notebook in `joel/kaggle_sumissions/submission4`.

## Setup

In [1]:
# Uncomment this in a fresh environment if these packages are missing.
# %pip install -q polars pyarrow xgboost prophet

from pathlib import Path
import gc
import math
import warnings

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.model_selection import StratifiedGroupKFold, StratifiedShuffleSplit
from prophet import Prophet
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

RANDOM_STATE = 90
REBUILD_INTERMEDIATES = False
RUN_QA_SUMMARIES = False
BUILD_SIMPLE_SAMPLE = False
USE_PROPHET_AS_MODEL_FEATURES = False
PROPHET_BLEND_STRENGTH = 0.08
PROPHET_FEATURE_COLUMNS = [
    'prophet_orders_yhat',
    'prophet_orders_trend',
    'prophet_orders_weekly',
    'prophet_orders_yearly',
    'prophet_orders_holidays',
]


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'joel').exists() and (candidate / 'charlie').exists():
            return candidate
    return Path.cwd()


def find_raw_train_path(project_root: Path) -> Path:
    candidates = [
        project_root / 'joel' / 'dat_train1.csv',
        project_root / 'data' / 'dat_train1.csv',
        Path.cwd() / 'dat_train1.csv',
        Path.cwd().parent / 'dat_train1.csv',
    ]
    for path in candidates:
        if path.exists():
            return path

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        matches = list(kaggle_input.rglob('dat_train1.csv'))
        if matches:
            return matches[0]

    raise FileNotFoundError('Could not find dat_train1.csv')


PROJECT_ROOT = find_project_root()
SUBMISSION_DIR = PROJECT_ROOT / 'joel' / 'kaggle_sumissions' / 'submission4'
if not SUBMISSION_DIR.exists():
    SUBMISSION_DIR = Path.cwd()

DATA_DIR = SUBMISSION_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = find_raw_train_path(PROJECT_ROOT)
TEST_EVENTS_PATH = SUBMISSION_DIR / 'open_journeys2.csv'
TEST_TEMPLATE_PATH = SUBMISSION_DIR / 'open_journeys2_flattened_all0.csv'
DT_CLEAN_PATH = DATA_DIR / 'dt_clean.parquet'
USER_OUTCOMES_PATH = DATA_DIR / 'user_outcomes.parquet'
TEST_FEATURES_PATH = DATA_DIR / (
    'test_features_open_journeys2.parquet' if TEST_EVENTS_PATH.exists() else 'test_features.parquet'
)
TRAIN_SIMPLE_PATH = DATA_DIR / 'train_features_simple_sample.parquet'
TRAIN_KCUT_PATH = DATA_DIR / 'train_features_kcut_sample.parquet'
PROPHET_DAILY_PATH = SUBMISSION_DIR / 'order_shipped_ts.csv'
PROPHET_FORECAST_PATH = DATA_DIR / 'prophet_daily_order_forecast.csv'
PROPHET_WEEKLY_FORECAST_PATH = DATA_DIR / 'prophet_weekly_order_forecast.csv'
PROPHET_MONTHLY_FORECAST_PATH = DATA_DIR / 'prophet_monthly_order_forecast.csv'
PROPHET_TRAINING_SUMMARY_PATH = DATA_DIR / 'prophet_training_summary.csv'
PROPHET_PLOT_DIR = SUBMISSION_DIR / 'prophet-plots'
PROPHET_PLOT_DIR.mkdir(parents=True, exist_ok=True)
SUB1_PATH = SUBMISSION_DIR / 'sub1_xgb_dev.csv'
SUB2_PATH = SUBMISSION_DIR / 'sub2_xgb_tuned_full.csv'
FINAL_SUBMISSION_PATH = SUBMISSION_DIR / 'submission.csv'

print('Project root:', PROJECT_ROOT)
print('Raw train path:', TRAIN_PATH)
print('Prediction events path:', TEST_EVENTS_PATH if TEST_EVENTS_PATH.exists() else 'training incomplete users')
print('Submission id template:', TEST_TEMPLATE_PATH if TEST_TEMPLATE_PATH.exists() else 'test feature order')
print('Output dir:', SUBMISSION_DIR)

Project root: /Users/joelyoon/Documents/git_repo/m148-project
Raw train path: /Users/joelyoon/Documents/git_repo/m148-project/joel/dat_train1.csv
Prediction events path: /Users/joelyoon/Documents/git_repo/m148-project/joel/kaggle_sumissions/submission4/open_journeys2.csv
Submission id template: /Users/joelyoon/Documents/git_repo/m148-project/joel/kaggle_sumissions/submission4/open_journeys2_flattened_all0.csv
Output dir: /Users/joelyoon/Documents/git_repo/m148-project/joel/kaggle_sumissions/submission4


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 01 Data Cleaning

In [2]:
def scan_events_csv(path: Path) -> pl.LazyFrame:
    lf = pl.scan_csv(path, infer_schema_length=20_000)
    schema = dict(lf.collect_schema())
    if schema.get('event_timestamp') == pl.String:
        lf = lf.with_columns(
            pl.col('event_timestamp')
            .str.to_datetime(format='%Y-%m-%dT%H:%M:%SZ', time_zone='UTC', strict=False)
            .alias('event_timestamp')
        )
    return lf


def write_lazy_parquet(lf: pl.LazyFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        lf.sink_parquet(path)
    except Exception:
        lf.collect(streaming=True).write_parquet(path)


raw_events = scan_events_csv(TRAIN_PATH)
schema_names = raw_events.collect_schema().names()

if RUN_QA_SUMMARIES:
    duplicate_event_count = (
        raw_events
        .group_by(['id', 'event_timestamp', 'event_name'])
        .agg(pl.len().alias('n'))
        .filter(pl.col('n') > 1)
        .select((pl.col('n') - 1).sum().alias('duplicates'))
        .collect(streaming=True)
        .item()
    )
    print('Duplicate id/timestamp/event rows:', duplicate_event_count)

    n_raw_rows = raw_events.select(pl.len()).collect(streaming=True).item()
    missing_counts = raw_events.select(
        [pl.col(col).is_null().sum().alias(col) for col in schema_names]
    ).collect(streaming=True)
    missing_summary = pd.DataFrame({
        'column': missing_counts.columns,
        'na_count': missing_counts.row(0),
    })
    missing_summary['pct_missing'] = (missing_summary['na_count'] / n_raw_rows * 100).round(2)
    display(missing_summary)

schema_names = set(schema_names)
drop_cols = [
    col for col in ['customer_id', 'account_id', 'journey_steps_until_end', 'sep']
    if col in schema_names
]

dt_clean_lf = (
    raw_events
    .unique(subset=['id', 'event_timestamp', 'event_name'], keep='first')
    .sort(['id', 'event_timestamp'])
    .drop(drop_cols)
)

if REBUILD_INTERMEDIATES or not DT_CLEAN_PATH.exists():
    write_lazy_parquet(dt_clean_lf, DT_CLEAN_PATH)

dt_clean = pl.scan_parquet(DT_CLEAN_PATH)
train_cutoff_ts = dt_clean.select(pl.max('event_timestamp')).collect().item()

user_outcomes_lf = (
    dt_clean
    .group_by('id')
    .agg(
        (pl.col('ed_id') == 28).any().alias('has_shipped'),
        pl.max('event_timestamp').alias('last_ts'),
    )
    .with_columns(
        ((pl.lit(train_cutoff_ts) - pl.col('last_ts')).dt.total_seconds() // 86_400)
        .cast(pl.Int32)
        .alias('days_inactive')
    )
    .with_columns(
        pl.when(pl.col('has_shipped'))
        .then(pl.lit('success'))
        .when(pl.col('days_inactive') >= 60)
        .then(pl.lit('failure'))
        .otherwise(pl.lit('incomplete'))
        .alias('final_outcome')
    )
    .select(['id', 'final_outcome'])
)

if REBUILD_INTERMEDIATES or not USER_OUTCOMES_PATH.exists():
    write_lazy_parquet(user_outcomes_lf, USER_OUTCOMES_PATH)

user_outcomes = pl.read_parquet(USER_OUTCOMES_PATH)
print(user_outcomes.group_by('final_outcome').len().sort('final_outcome'))

del raw_events, dt_clean_lf, user_outcomes_lf
gc.collect()

shape: (3, 2)
┌───────────────┬────────┐
│ final_outcome ┆ len    │
│ ---           ┆ ---    │
│ str           ┆ u32    │
╞═══════════════╪════════╡
│ failure       ┆ 992757 │
│ incomplete    ┆ 158325 │
│ success       ┆ 279363 │
└───────────────┴────────┘


40

## 02 Sampling and Flattening

In [3]:
def ts_seconds(column: str) -> pl.Expr:
    return pl.col(column).dt.timestamp('us') // 1_000_000


def flatten_journey(events: pl.DataFrame, id_col: str = 'id') -> pl.DataFrame:
    events = (
        events
        .sort([id_col, 'event_timestamp'])
        .with_columns(
            ts_seconds('event_timestamp').alias('_ts_s'),
            ts_seconds('cutoff_time').alias('_cutoff_s'),
        )
        .with_columns(
            (pl.col('_ts_s') - pl.col('_ts_s').shift(1).over(id_col)).alias('_gap_s')
        )
    )

    summary = events.group_by(id_col).agg(
        pl.first('event_timestamp').alias('first_action_ts'),
        pl.last('event_timestamp').alias('last_action_ts'),
        (pl.last('_ts_s') - pl.first('_ts_s')).alias('journey_length_s'),
        ((pl.first('_cutoff_s') - pl.last('_ts_s')) // 86_400).cast(pl.Int32).alias('days_inactive'),
        pl.len().alias('total_actions'),
        pl.col('_gap_s').mean().alias('mean_gap_sec'),
        pl.col('_gap_s').median().alias('median_gap_sec'),
        pl.col('_gap_s').max().alias('max_gap_sec'),
    )

    counts_long = events.group_by([id_col, 'event_name']).agg(pl.len().alias('count'))
    counts = counts_long.pivot(
        index=id_col,
        columns='event_name',
        values='count',
        aggregate_function='first',
    ).fill_null(0)
    counts = counts.rename({col: f'n_{col}' for col in counts.columns if col != id_col})

    return summary.join(counts, on=id_col, how='left')


dt_clean = pl.scan_parquet(DT_CLEAN_PATH)
user_outcomes = pl.read_parquet(USER_OUTCOMES_PATH)
using_external_test = TEST_EVENTS_PATH.exists()
if using_external_test:
    test_ids = scan_events_csv(TEST_EVENTS_PATH).select('id').unique().collect()
else:
    test_ids = user_outcomes.filter(pl.col('final_outcome') == 'incomplete').select('id')
train_ids = user_outcomes.filter(pl.col('final_outcome') != 'incomplete').select('id')

print('Train ids:', train_ids.height)
print('Prediction source:', TEST_EVENTS_PATH.name if using_external_test else 'incomplete users from training data')
print('Test/open ids:', test_ids.height)

Train ids: 1272120
Prediction source: open_journeys2.csv
Test/open ids: 123467


In [4]:
# Flatten the external open journeys when present; otherwise use incomplete training journeys.
if REBUILD_INTERMEDIATES or not TEST_FEATURES_PATH.exists():
    if using_external_test:
        test_events_lf = scan_events_csv(TEST_EVENTS_PATH)
        test_schema = set(test_events_lf.collect_schema().names())
        test_drop_cols = [
            col for col in ['customer_id', 'account_id', 'journey_steps_until_end', 'sep']
            if col in test_schema
        ]
        test_events_lf = (
            test_events_lf
            .unique(subset=['id', 'event_timestamp', 'event_name'], keep='first')
            .sort(['id', 'event_timestamp'])
            .drop(test_drop_cols)
        )
        test_cutoff = test_events_lf.select(pl.max('event_timestamp')).collect().item()
        test_events = test_events_lf.with_columns(pl.lit(test_cutoff).alias('cutoff_time')).collect()
    else:
        test_cutoff = (
            dt_clean
            .join(test_ids.lazy(), on='id', how='inner')
            .select(pl.max('event_timestamp'))
            .collect()
            .item()
        )
        test_events = (
            dt_clean
            .join(test_ids.lazy(), on='id', how='inner')
            .with_columns(pl.lit(test_cutoff).alias('cutoff_time'))
            .collect()
        )
    test_flat = flatten_journey(test_events).with_columns(pl.lit('incomplete').alias('final_outcome'))
    test_flat.write_parquet(TEST_FEATURES_PATH)

    del test_events, test_flat
    gc.collect()

pl.read_parquet(TEST_FEATURES_PATH).head()

id,first_action_ts,last_action_ts,journey_length_s,days_inactive,total_actions,mean_gap_sec,median_gap_sec,max_gap_sec,n_view_cart,n_promotion_created,n_campaign_click,n_add_to_cart,n_application_web_submit,n_account_activitation,n_application_web_view,n_application_web_approved,n_browse_products,n_application_web_declined,n_campaignemail_clicked,n_begin_checkout,n_catalog_(mail),n_pre-application_(3rd_party_affiliates),n_application_phone_approved,n_place_order_web,n_site_registration,n_place_downpayment,n_account_downpaymentcleared,n_place_order_phone,n_application_phone_declined,n_catalog_(email)_(experian),n_account_downpaymentreceived,n_fingerhut_university,final_outcome
str,"datetime[μs, UTC]","datetime[μs, UTC]",i64,i32,u32,f64,f64,i64,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,str
"""-1201406952 -1898198331""",2023-04-25 10:25:32 UTC,2023-04-25 16:25:32 UTC,21600,27,6,4320.0,1948.0,15585,0,1,1,0,1,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,"""incomplete"""
"""-234582835 335133819""",2023-05-01 17:41:23 UTC,2023-05-01 23:41:23 UTC,21600,21,4,7200.0,3765.0,17835,0,0,1,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""incomplete"""
"""-2082502459 1736950868""",2022-11-17 10:34:24 UTC,2023-05-15 20:37:36 UTC,15501792,7,88,178181.517241,73.0,6648638,16,3,1,5,2,0,15,1,44,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,"""incomplete"""
"""1679653819 1158547752""",2023-01-30 07:45:26 UTC,2023-05-09 14:20:49 UTC,8577323,13,35,252274.205882,0.0,2643624,6,4,0,4,3,0,4,1,9,1,0,3,0,0,0,0,0,0,0,0,0,0,0,0,"""incomplete"""
"""1983285849 2042326961""",2023-05-04 05:52:31 UTC,2023-05-04 10:50:37 UTC,17886,19,3,8943.0,8943.0,17886,0,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""incomplete"""


### Simple one-cutoff sample

In [5]:
def add_random_cutoff(df: pl.DataFrame, seed: int) -> pl.DataFrame:
    rng = np.random.default_rng(seed)
    df = df.with_columns(
        (ts_seconds('end_ts') - ts_seconds('start_ts')).cast(pl.Float64).alias('_duration_s'),
        pl.Series('_u', rng.random(df.height)),
    )
    return df.with_columns(
        (
            pl.col('start_ts')
            + pl.duration(seconds=(pl.col('_duration_s') * pl.col('_u')).round().cast(pl.Int64))
        ).alias('random_cutoff')
    ).drop(['_duration_s', '_u'])


if BUILD_SIMPLE_SAMPLE and (REBUILD_INTERMEDIATES or not TRAIN_SIMPLE_PATH.exists()):
    train_events = (
        dt_clean
        .join(train_ids.lazy(), on='id', how='inner')
        .collect(streaming=True)
    )

    cutoffs = train_events.group_by('id').agg(
        pl.min('event_timestamp').alias('start_ts'),
        pl.max('event_timestamp').alias('end_ts'),
    )
    cutoffs = add_random_cutoff(cutoffs, seed=42)

    train_events_truncated = (
        train_events
        .join(cutoffs.select(['id', 'random_cutoff']), on='id', how='inner')
        .filter(pl.col('event_timestamp') <= pl.col('random_cutoff'))
        .rename({'random_cutoff': 'cutoff_time'})
    )

    train_flat_simple = flatten_journey(train_events_truncated)
    train_flat_simple = train_flat_simple.join(
        user_outcomes.filter(pl.col('final_outcome') != 'incomplete'),
        on='id',
        how='inner',
    )
    train_flat_simple.write_parquet(TRAIN_SIMPLE_PATH)

    del train_events, train_events_truncated, train_flat_simple, cutoffs
    gc.collect()

if TRAIN_SIMPLE_PATH.exists():
    print(pl.scan_parquet(TRAIN_SIMPLE_PATH).select(pl.len()).collect())

### K cutoffs, proportional to total actions

In [6]:
def make_k_cutoffs_by_total_actions(train_events: pl.DataFrame, seed: int = 42) -> pl.DataFrame:
    snapshot_counts = train_events.group_by('id').agg(
        pl.len().alias('total_actions'),
        pl.min('event_timestamp').alias('start_ts'),
        pl.max('event_timestamp').alias('end_ts'),
    )
    snapshot_counts = snapshot_counts.join(
        user_outcomes.select(['id', 'final_outcome']),
        on='id',
        how='left',
    )

    n_users = snapshot_counts.height
    snapshot_counts = snapshot_counts.with_columns(
        ((5 * pl.col('total_actions').rank(method='min') / n_users).ceil())
        .clip(1, 5)
        .cast(pl.Int64)
        .alias('K')
    )

    cutoffs = (
        snapshot_counts
        .with_columns(pl.int_ranges(1, pl.col('K') + 1).alias('_snapshot_n'))
        .explode('_snapshot_n')
    )
    cutoffs = add_random_cutoff(cutoffs, seed=seed)
    return cutoffs.with_columns(
        (pl.col('id').cast(pl.Utf8) + '_' + pl.col('_snapshot_n').cast(pl.Utf8)).alias('snapshot_id')
    ).select(['id', 'snapshot_id', 'random_cutoff'])


# Charlie's final code overwrites these total-action cutoffs with the journey-days cutoffs below.
# To inspect them, collect train_events and call make_k_cutoffs_by_total_actions(train_events).

### K cutoffs, proportional to journey days

In [7]:
def make_k_cutoffs(train_events: pl.DataFrame, seed: int = 13) -> pl.DataFrame:
    snapshot_counts = train_events.group_by('id').agg(
        pl.len().alias('total_actions'),
        pl.min('event_timestamp').alias('start_ts'),
        pl.max('event_timestamp').alias('end_ts'),
    )
    snapshot_counts = snapshot_counts.with_columns(
        ((ts_seconds('end_ts') - ts_seconds('start_ts')) / 86_400).alias('journey_days')
    )
    snapshot_counts = snapshot_counts.join(
        user_outcomes.select(['id', 'final_outcome']),
        on='id',
        how='left',
    )

    n_users = snapshot_counts.height
    snapshot_counts = snapshot_counts.with_columns(
        ((5 * pl.col('journey_days').rank(method='min') / n_users).ceil())
        .clip(1, 5)
        .cast(pl.Int64)
        .alias('K')
    )

    cutoffs = (
        snapshot_counts
        .with_columns(pl.int_ranges(1, pl.col('K') + 1).alias('_snapshot_n'))
        .explode('_snapshot_n')
    )
    cutoffs = add_random_cutoff(cutoffs, seed=seed)
    cutoffs = cutoffs.with_columns(
        (pl.col('id').cast(pl.Utf8) + '_' + pl.col('_snapshot_n').cast(pl.Utf8)).alias('snapshot_id')
    )
    return cutoffs.select(['id', 'snapshot_id', 'random_cutoff'])


if REBUILD_INTERMEDIATES or not TRAIN_KCUT_PATH.exists():
    train_events = (
        dt_clean
        .join(train_ids.lazy(), on='id', how='inner')
        .collect(streaming=True)
    )
    cutoffs = make_k_cutoffs(train_events, seed=13)

    train_events_expanded = (
        train_events
        .join(cutoffs, on='id', how='inner')
        .filter(pl.col('event_timestamp') <= pl.col('random_cutoff'))
        .rename({'id': 'original_user_id', 'snapshot_id': 'id', 'random_cutoff': 'cutoff_time'})
    )

    train_flat = flatten_journey(train_events_expanded)
    train_flat = (
        train_flat
        .rename({'id': 'snapshot_id'})
        .with_columns(pl.col('snapshot_id').str.replace(r'_[0-9]+$', '').alias('id'))
        .join(
            user_outcomes.filter(pl.col('final_outcome') != 'incomplete'),
            on='id',
            how='inner',
        )
    )
    train_flat.write_parquet(TRAIN_KCUT_PATH)

    del train_events, cutoffs, train_events_expanded, train_flat
    gc.collect()

print(pl.scan_parquet(TRAIN_KCUT_PATH).select(pl.len()).collect())

shape: (1, 1)
┌─────────┐
│ len     │
│ ---     │
│ u32     │
╞═════════╡
│ 3815408 │
└─────────┘


## 03 Modelling

In [8]:
train_flat = pl.read_parquet(TRAIN_KCUT_PATH).to_pandas()
test_flat = pl.read_parquet(TEST_FEATURES_PATH).to_pandas()
test_flat['id'] = test_flat['id'].astype(str)

if TEST_TEMPLATE_PATH.exists():
    submission_ids = pd.read_csv(TEST_TEMPLATE_PATH, usecols=['id'])['id'].astype(str)
    test_flat = pd.DataFrame({'id': submission_ids}).merge(test_flat, on='id', how='left')
else:
    submission_ids = test_flat['id'].astype(str)


def normalize_day(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, utc=True, errors='coerce').dt.tz_convert(None).dt.floor('D')


def get_success_journey_cutoff_days(default: int = 71) -> int:
    try:
        success_ids = (
            pl.scan_parquet(USER_OUTCOMES_PATH)
            .filter(pl.col('final_outcome') == 'success')
            .select('id')
        )
        p80 = (
            pl.scan_parquet(DT_CLEAN_PATH)
            .join(success_ids, on='id', how='inner')
            .group_by('id')
            .agg(
                pl.min('event_timestamp').alias('first_ts'),
                pl.max('event_timestamp').alias('last_ts'),
            )
            .with_columns(
                (pl.col('last_ts') - pl.col('first_ts')).dt.total_days().alias('journey_length_days')
            )
            .select(pl.col('journey_length_days').quantile(0.80).alias('p80_days'))
            .collect()
            .item()
        )
        if pd.notna(p80) and np.isfinite(p80):
            return int(round(float(p80)))
    except Exception as exc:
        print(f'Using default Prophet start cutoff ({default} days): {exc}')
    return default


def load_prophet_daily_counts() -> pd.DataFrame:
    if PROPHET_DAILY_PATH.exists():
        daily = pd.read_csv(PROPHET_DAILY_PATH)
        daily = daily.rename(columns={'date': 'ds', 'n_order_shipped': 'y'})[['ds', 'y']]
        daily['ds'] = pd.to_datetime(daily['ds'])
        return daily.sort_values('ds').reset_index(drop=True)

    daily = (
        pl.scan_parquet(DT_CLEAN_PATH)
        .with_columns(pl.col('event_timestamp').dt.date().alias('ds'))
        .group_by('ds')
        .agg((pl.col('event_name') == 'order_shipped').sum().alias('y'))
        .sort('ds')
        .collect()
        .to_pandas()
    )
    daily['ds'] = pd.to_datetime(daily['ds'])
    return daily


def fit_prophet_model(train_data: pd.DataFrame, *, yearly=True, weekly=False, holidays=False) -> Prophet:
    model = Prophet(
        yearly_seasonality=yearly,
        weekly_seasonality=weekly,
        daily_seasonality=False,
    )
    if holidays:
        try:
            model.add_country_holidays(country_name='US')
        except Exception as exc:
            print('Skipping Prophet US holidays:', exc)
    return model.fit(train_data)


def save_prophet_plots(model: Prophet, forecast: pd.DataFrame, name: str) -> None:
    fig = model.plot(forecast)
    fig.savefig(PROPHET_PLOT_DIR / f'{name}_forecast.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

    fig = model.plot_components(forecast)
    fig.savefig(PROPHET_PLOT_DIR / f'{name}_components.png', dpi=150, bbox_inches='tight')
    plt.close(fig)


def fit_prophet_order_features(frames: list[pd.DataFrame]) -> pd.DataFrame:
    prophet_daily = load_prophet_daily_counts()
    cutoff_days = get_success_journey_cutoff_days(default=71)
    start_cutoff = prophet_daily['ds'].min() + pd.Timedelta(days=cutoff_days)

    prophet_train_data = prophet_daily[prophet_daily['ds'] > start_cutoff].copy()
    daily_train_data = prophet_train_data[prophet_train_data['ds'].dt.dayofweek < 5].copy()

    weekly_train_data = prophet_train_data.copy()
    weekly_train_data['ds'] = weekly_train_data['ds'].dt.to_period('W').apply(lambda period: period.start_time)
    weekly_train_data = weekly_train_data.groupby('ds', as_index=False)['y'].sum().sort_values('ds')

    monthly_train_data = prophet_train_data.copy()
    monthly_train_data['ds'] = monthly_train_data['ds'].dt.to_period('M').dt.to_timestamp()
    monthly_train_data = monthly_train_data.groupby('ds', as_index=False)['y'].sum().sort_values('ds')
    monthly_train_data = monthly_train_data[
        (monthly_train_data['ds'] > pd.Timestamp('2021-01-01'))
        & (monthly_train_data['ds'] < pd.Timestamp('2023-01-01'))
    ].copy()

    needed_days = pd.concat(
        [normalize_day(frame['last_action_ts']) for frame in frames],
        ignore_index=True,
    ).dropna()
    forecast_end = max(pd.Timestamp('2024-12-01'), prophet_train_data['ds'].max(), needed_days.max())

    summary = pd.DataFrame([
        {'key': 'success_journey_length_p80_days', 'value': cutoff_days},
        {'key': 'prophet_start_cutoff', 'value': start_cutoff.date().isoformat()},
        {'key': 'daily_train_rows_weekdays_only', 'value': len(daily_train_data)},
        {'key': 'weekly_train_rows', 'value': len(weekly_train_data)},
        {'key': 'monthly_train_rows', 'value': len(monthly_train_data)},
        {'key': 'forecast_end', 'value': forecast_end.date().isoformat()},
    ])
    summary.to_csv(PROPHET_TRAINING_SUMMARY_PATH, index=False)

    daily_model = fit_prophet_model(daily_train_data, yearly=True, weekly=True, holidays=True)
    daily_future = pd.DataFrame({'ds': pd.date_range(daily_train_data['ds'].min(), forecast_end, freq='D')})
    daily_forecast_raw = daily_model.predict(daily_future)
    if 'holidays' not in daily_forecast_raw.columns:
        daily_forecast_raw['holidays'] = 0.0
    daily_forecast_raw.to_csv(PROPHET_FORECAST_PATH, index=False)
    save_prophet_plots(daily_model, daily_forecast_raw, 'daily_orders')

    weekly_model = fit_prophet_model(weekly_train_data, yearly=True, weekly=False, holidays=False)
    weekly_future = pd.DataFrame({'ds': pd.date_range(weekly_train_data['ds'].min(), forecast_end, freq='W')})
    weekly_forecast = weekly_model.predict(weekly_future)
    weekly_forecast.to_csv(PROPHET_WEEKLY_FORECAST_PATH, index=False)
    save_prophet_plots(weekly_model, weekly_forecast, 'weekly_orders')

    monthly_model = fit_prophet_model(monthly_train_data, yearly=True, weekly=False, holidays=False)
    monthly_future = pd.DataFrame({'ds': pd.date_range(monthly_train_data['ds'].min(), forecast_end, freq='MS')})
    monthly_forecast = monthly_model.predict(monthly_future)
    monthly_forecast.to_csv(PROPHET_MONTHLY_FORECAST_PATH, index=False)
    save_prophet_plots(monthly_model, monthly_forecast, 'monthly_orders')

    forecast = daily_forecast_raw[['ds', 'yhat', 'trend', 'weekly', 'yearly', 'holidays']].rename(columns={
        'ds': 'prophet_ds',
        'yhat': 'prophet_orders_yhat',
        'trend': 'prophet_orders_trend',
        'weekly': 'prophet_orders_weekly',
        'yearly': 'prophet_orders_yearly',
        'holidays': 'prophet_orders_holidays',
    })
    print('Prophet summary')
    display(summary)
    return forecast


def add_prophet_features(frame: pd.DataFrame, prophet_forecast: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame['prophet_ds'] = normalize_day(frame['last_action_ts'])
    frame = frame.merge(prophet_forecast, on='prophet_ds', how='left')
    prophet_cols = [col for col in prophet_forecast.columns if col != 'prophet_ds']
    frame[prophet_cols] = frame[prophet_cols].fillna(frame[prophet_cols].median()).fillna(0)
    return frame


prophet_forecast = fit_prophet_order_features([train_flat, test_flat])
train_flat = add_prophet_features(train_flat, prophet_forecast)
test_flat = add_prophet_features(test_flat, prophet_forecast)
print('Prophet features added:', [col for col in prophet_forecast.columns if col != 'prophet_ds'])

train_flat['final_outcome'] = pd.Categorical(
    train_flat['final_outcome'], categories=['success', 'failure']
)
test_flat['final_outcome'] = test_flat['final_outcome'].fillna('incomplete')
test_flat['final_outcome'] = pd.Categorical(
    test_flat['final_outcome'], categories=['success', 'failure']
)

# Match Charlie's physical downsampling: keep failures and sample successes to a 1:19 success:failure ratio.
successes = train_flat[train_flat['final_outcome'] == 'success']
failures = train_flat[train_flat['final_outcome'] == 'failure']
n_target = min(math.floor(len(failures) / 19), len(successes))

train_flat_balanced = pd.concat(
    [failures, successes.sample(n=n_target, random_state=RANDOM_STATE)],
    ignore_index=True,
)

train_dev = (
    train_flat_balanced
    .groupby('final_outcome', observed=True, group_keys=False)
    .sample(frac=0.10, random_state=RANDOM_STATE)
)

print(train_flat_balanced['final_outcome'].value_counts())
print('Dev rows:', len(train_dev))

del train_flat, successes, failures
gc.collect()

15:45:37 - cmdstanpy - INFO - Chain [1] start processing


15:45:37 - cmdstanpy - INFO - Chain [1] done processing


15:45:37 - cmdstanpy - INFO - Chain [1] start processing


15:45:37 - cmdstanpy - INFO - Chain [1] done processing


15:45:37 - cmdstanpy - INFO - Chain [1] start processing


...

Prophet summary


,key,value
0,success_journey_length_p80_days,71
1,prophet_start_cutoff,2021-04-13
2,daily_train_rows_weekdays_only,636
3,weekly_train_rows,128
4,monthly_train_rows,21
5,forecast_end,2024-12-01


Prophet features added: ['prophet_orders_yhat', 'prophet_orders_trend', 'prophet_orders_weekly', 'prophet_orders_yearly', 'prophet_orders_holidays']


final_outcome
failure    3264641
success     171823
Name: count, dtype: int64
Dev rows: 343646


244

In [9]:
LEAKERS = [
    'first_action_ts',
    'last_action_ts',
    'n_order_shipped',
    'n_place_order_web',
    'n_place_order_phone',
    'n_place_downpayment',
    'n_account_downpaymentreceived',
    'n_account_downpaymentcleared',
    'n_customer_requested_catalog_(digital)',
]


def make_xy(frame: pd.DataFrame, feature_columns: list[str] | None = None):
    y = (frame['final_outcome'] == 'success').astype(int)
    groups = frame['id'].astype(str)

    prophet_feature_drop_cols = [] if USE_PROPHET_AS_MODEL_FEATURES else PROPHET_FEATURE_COLUMNS
    drop_cols = {'final_outcome', 'id', 'snapshot_id', 'prophet_ds', *LEAKERS, *prophet_feature_drop_cols}
    if feature_columns is None:
        feature_columns = [col for col in frame.columns if col not in drop_cols]

    X = frame.reindex(columns=feature_columns, fill_value=0).copy()
    for col in X.select_dtypes(include=['datetime64[ns]', 'datetimetz']).columns:
        X[col] = X[col].view('int64') / 1e9
    X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
    return X, y, groups, feature_columns


def evaluate_cv(model, X, y, groups, folds: int = 5) -> pd.DataFrame:
    splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    for fold, (train_idx, valid_idx) in enumerate(splitter.split(X, y, groups), start=1):
        fold_model = clone(model)
        fold_model.fit(X.iloc[train_idx], y.iloc[train_idx])
        prob = fold_model.predict_proba(X.iloc[valid_idx])[:, 1]
        rows.append({
            'fold': fold,
            'bbrier': brier_score_loss(y.iloc[valid_idx], prob),
            'prauc': average_precision_score(y.iloc[valid_idx], prob),
        })
    return pd.DataFrame(rows)


def apply_prophet_adjustment(base_prob, scored_frame: pd.DataFrame, reference_frame: pd.DataFrame) -> np.ndarray:
    signal = scored_frame['prophet_orders_yhat']
    reference = reference_frame['prophet_orders_yhat']
    iqr = reference.quantile(0.75) - reference.quantile(0.25)
    if not np.isfinite(iqr) or iqr == 0:
        iqr = reference.std()
    if not np.isfinite(iqr) or iqr == 0:
        return np.asarray(base_prob)

    z = ((signal - reference.median()) / iqr).clip(-2, 2).fillna(0)
    multiplier = np.exp(PROPHET_BLEND_STRENGTH * z)
    adjusted = np.asarray(base_prob) * multiplier.to_numpy()
    adjusted *= np.asarray(base_prob).mean() / adjusted.mean()
    return np.clip(adjusted, 0, 1)


X_train, y_train, groups_train, feature_columns = make_xy(train_flat_balanced)
X_dev, y_dev, groups_dev, _ = make_xy(train_dev, feature_columns)
X_test, _, _, _ = make_xy(test_flat, feature_columns)

print('Train matrix:', X_train.shape)
print('Dev matrix:', X_dev.shape)
print('Test matrix:', X_test.shape)

Train matrix: (3436464, 26)
Dev matrix: (343646, 26)
Test matrix: (123467, 26)


In [10]:
rf_base = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

xgb_base = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    booster='gbtree',
    tree_method='hist',
    objective='binary:logistic',
    eval_metric='logloss',
    n_jobs=8,
    random_state=RANDOM_STATE,
)

xgb_dev_cv = evaluate_cv(xgb_base, X_dev, y_dev, groups_dev, folds=5)
rf_dev_cv = evaluate_cv(rf_base, X_dev, y_dev, groups_dev, folds=5)

print('XGB dev CV')
display(xgb_dev_cv.agg({'bbrier': ['mean', 'std'], 'prauc': ['mean', 'std']}))
print('RF dev CV')
display(rf_dev_cv.agg({'bbrier': ['mean', 'std'], 'prauc': ['mean', 'std']}))

xgb_base.fit(X_dev, y_dev)
sub1_prob = xgb_base.predict_proba(X_test)[:, 1]
sub1_prob = apply_prophet_adjustment(sub1_prob, test_flat, train_flat_balanced)
sub1 = pd.DataFrame({
    'id': submission_ids,
    'order_shipped': sub1_prob,
})
sub1.to_csv(SUB1_PATH, index=False)
SUB1_PATH

XGB dev CV


,bbrier,prauc
mean,0.036697,0.412109
std,0.000750,0.007958


RF dev CV


,bbrier,prauc
mean,0.036917,0.406853
std,0.000712,0.007168


PosixPath('/Users/joelyoon/Documents/git_repo/m148-project/joel/kaggle_sumissions/submission4/sub1_xgb_dev.csv')

In [11]:
xgb_full_cv = evaluate_cv(xgb_base, X_train, y_train, groups_train, folds=5)
display(xgb_full_cv.agg({'bbrier': ['mean', 'std'], 'prauc': ['mean', 'std']}))

,bbrier,prauc
mean,0.036461,0.418685
std,0.000086,0.001009


In [12]:
def tune_xgb_holdout(X: pd.DataFrame, y: pd.Series, groups: pd.Series, n_iter: int = 15):
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    tune_train_idx, tune_valid_idx = next(splitter.split(X, y, groups))

    rng = np.random.default_rng(RANDOM_STATE)
    results = []
    best_score = np.inf
    best_params = None

    for i in range(n_iter):
        params = {
            'n_estimators': int(rng.integers(50, 301)),
            'learning_rate': float(rng.uniform(0.05, 0.30)),
            'max_depth': int(rng.integers(3, 9)),
        }
        model = XGBClassifier(
            **params,
            booster='gbtree',
            tree_method='hist',
            objective='binary:logistic',
            eval_metric='logloss',
            n_jobs=8,
            random_state=RANDOM_STATE,
        )
        model.fit(X.iloc[tune_train_idx], y.iloc[tune_train_idx])
        prob = model.predict_proba(X.iloc[tune_valid_idx])[:, 1]
        score = brier_score_loss(y.iloc[tune_valid_idx], prob)
        row = {'iteration': i + 1, 'bbrier': score, **params}
        results.append(row)
        if score < best_score:
            best_score = score
            best_params = params

    final_model = XGBClassifier(
        **best_params,
        booster='gbtree',
        tree_method='hist',
        objective='binary:logistic',
        eval_metric='logloss',
        n_jobs=8,
        random_state=RANDOM_STATE,
    )
    final_model.fit(X, y)
    return final_model, pd.DataFrame(results).sort_values('bbrier')


xgb_tuned, tuning_results = tune_xgb_holdout(X_train, y_train, groups_train, n_iter=15)
display(tuning_results)

sub2_prob = xgb_tuned.predict_proba(X_test)[:, 1]
sub2_prob = apply_prophet_adjustment(sub2_prob, test_flat, train_flat_balanced)
sub2 = pd.DataFrame({
    'id': submission_ids,
    'order_shipped': sub2_prob,
})
sub2.to_csv(SUB2_PATH, index=False)
sub2.to_csv(FINAL_SUBMISSION_PATH, index=False)
SUB2_PATH, FINAL_SUBMISSION_PATH

,iteration,bbrier,n_estimators,learning_rate,max_depth
9,10,0.036280,281,0.080160,8
7,8,0.036308,225,0.118483,7
2,3,0.036335,181,0.081348,7
8,9,0.036338,227,0.065624,7
12,13,0.036355,254,0.267038,5
0,1,0.036386,214,0.220474,5
11,12,0.036391,218,0.276418,6
6,7,0.036427,155,0.216201,6
1,2,0.036439,107,0.154588,6
3,4,0.036443,169,0.264284,7


(PosixPath('/Users/joelyoon/Documents/git_repo/m148-project/joel/kaggle_sumissions/submission4/sub2_xgb_tuned_full.csv'),
 PosixPath('/Users/joelyoon/Documents/git_repo/m148-project/joel/kaggle_sumissions/submission4/submission.csv'))